# 01 — EDA: National Trends (2013–2024)

**Questions explored:**
1. How has tax revenue grown since devolution?
2. Are budget allocations keeping pace? What's the execution gap?
3. How fast is the debt burden growing relative to revenue?
4. Which sectors (education, health, infrastructure) get what share?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA = os.path.join("..", "data", "clean")
df = pd.read_csv(os.path.join(DATA, "national_enriched.csv"))
df.head()

## 1. Revenue Trend — Target vs Actual

> *Has KRA been hitting its targets? How fast is revenue growing?*

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

x = range(len(df))
ax1.plot(x, df["revenue_target_bn"], "o--", color="gray", label="Target")
ax1.plot(x, df["revenue_actual_bn"], "o-", color="#E24A33", linewidth=2, label="Actual")
ax1.fill_between(x, df["revenue_actual_bn"], df["revenue_target_bn"],
                 alpha=0.15, color="red", label="Shortfall")
ax1.set_xticks(x)
ax1.set_xticklabels(df["fiscal_year"], rotation=45, ha="right")
ax1.set_ylabel("KSh Billions")
ax1.set_title("KRA Revenue: Target vs Actual (2013/14 – 2023/24)")
ax1.legend(loc="upper left")

# Secondary axis: YoY growth
ax2 = ax1.twinx()
ax2.bar(x, df["revenue_actual_bn_growth_pct"], alpha=0.3, color="steelblue", width=0.4, label="YoY Growth %")
ax2.set_ylabel("YoY Growth %", color="steelblue")
ax2.axhline(0, color="steelblue", linewidth=0.5)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

# Revenue compliance gap
print("Revenue Compliance Gap (actual vs target, %):")
print(df[["fiscal_year", "revenue_compliance_gap_pct"]].to_string(index=False))

## 2. Budget Allocation vs Actual Expenditure

> *How much of the budget is actually spent? Which sectors have the worst execution?*

In [ ]:
sectors = ["education", "healthcare", "infrastructure", "total"]
colors = ["#348ABD", "#E24A33", "#988ED5", "#333333"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
for ax, sector, color in zip(axes.flat, sectors, colors):
    alloc_col = f"allocation_bn_{sector}"
    exp_col = f"expenditure_actual_bn_{sector}"
    if alloc_col in df.columns and exp_col in df.columns:
        ax.bar(x, df[alloc_col], alpha=0.4, color=color, label="Allocated")
        ax.bar(x, df[exp_col], alpha=0.8, color=color, width=0.5, label="Spent")
        ax.set_title(sector.title())
        ax.set_ylabel("KSh Bn")
        ax.legend(fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(df["fiscal_year"], rotation=45, ha="right", fontsize=8)

plt.suptitle("Budget Allocation vs Actual Expenditure by Sector", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Execution rate
if "budget_execution_rate_pct" in df.columns:
    print("Budget Execution Rate (%):")
    print(df[["fiscal_year", "budget_execution_rate_pct"]].to_string(index=False))

## 3. Sector Budget Shares Over Time

> *How has the allocation pie shifted between education, health, and infrastructure?*

In [ ]:
sector_cols = ["allocation_bn_education", "allocation_bn_healthcare", "allocation_bn_infrastructure"]
available = [c for c in sector_cols if c in df.columns]

if available:
    shares = df[available].div(df["allocation_bn_total"], axis=0) * 100
    shares.columns = [c.replace("allocation_bn_", "").title() for c in available]
    shares["fiscal_year"] = df["fiscal_year"]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    shares.set_index("fiscal_year").plot.area(ax=ax, alpha=0.7,
        color=["#348ABD", "#E24A33", "#988ED5"])
    ax.set_ylabel("% of Total Budget")
    ax.set_title("Sector Share of National Budget Over Time")
    ax.set_ylim(0, 100)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
    
    # Remaining share goes to "other"
    shares["Other"] = 100 - shares.drop(columns=["fiscal_year"], errors="ignore").sum(axis=1)
    display(shares.round(1))

## 4. Debt Burden — Stock, Servicing, and the Squeeze

> *How much of each shilling earned goes to debt? Is it accelerating?*

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

# Debt stock bars
ax1.bar(x, df["debt_stock_bn"], color="#988ED5", alpha=0.6, label="Debt Stock (KSh Bn)")
ax1.set_xticks(x)
ax1.set_xticklabels(df["fiscal_year"], rotation=45, ha="right")
ax1.set_ylabel("KSh Billions", color="#988ED5")
ax1.legend(loc="upper left")

# Debt-to-revenue ratio line
ax2 = ax1.twinx()
ax2.plot(x, df["debt_to_revenue_ratio"], "s-", color="#E24A33", linewidth=2, label="Debt / Revenue Ratio")
ax2.set_ylabel("Debt-to-Revenue Ratio", color="#E24A33")
ax2.legend(loc="center right")

ax1.set_title("Public Debt Growth & Debt-to-Revenue Ratio")
plt.tight_layout()
plt.show()

# Debt service as % of revenue
if "debt_service_bn" in df.columns:
    df["debt_service_pct_revenue"] = df["debt_service_bn"] / df["revenue_actual_bn"] * 100
    print("Debt Servicing as % of Revenue:")
    print(df[["fiscal_year", "debt_service_bn", "debt_service_pct_revenue"]].to_string(index=False))

## 5. Summary — National Picture

**Key takeaways** (to be confirmed when cells are run):

- Revenue has grown ~2× since 2013/14, but consistently misses target (compliance gap ~-4% to -8%).
- Budget has tripled, but execution rate hovers around 90–94% — ~KSh 100–200 Bn unspent per year.
- Debt stock has grown from KSh 2.1 Tn to ~10 Tn — debt-to-revenue ratio went from ~2× to ~5.5×.
- Debt servicing now consumes 30–50% of revenue, squeezing fiscal space for service delivery.
- Education gets the largest sector share (~20%), healthcare gets ~3–4%, infrastructure ~15–18%.

These patterns set the stage for the county-level question: *given constrained national resources, are the funds that do reach counties translating into better services?*